<a href="https://colab.research.google.com/github/LIBY70/Data-Analysis/blob/main/ida_week12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab12: 머신러닝 I — 분류

본 실습 자료는 『배워서 바로 써먹는 데이터 분석 with 파이썬』 (설진욱, 생능북스) 및
『파이썬을 활용한 빅데이터 분석개론』 (안기수, 생능출판사)의 내용을 참고하여 제작되었습니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    precision_recall_curve, roc_curve, auc, roc_auc_score
)

### 한글 폰트 설정

In [ ]:
!apt-get install -y fonts-nanum

In [ ]:
import platform

if platform.system() == 'Darwin':
    mpl.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    mpl.rc('font', family='Malgun Gothic')
else:
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    mpl.rc('font', family='NanumGothic')

mpl.rc('axes', unicode_minus=False)
print('폰트 설정 완료:', mpl.rcParams['font.family'])

---
## PART 1. 데이터 준비 — 유방암 진단 데이터

**데이터셋**: `sklearn.datasets.load_breast_cancer()`
- 표본: **569개** (악성 212개 / 정상 357개)
- 변수: **30개** 수치형 특성 (세포핵 반지름, 텍스처, 둘레, 면적 등)
- 목표: 세포 측정값으로 악성(malignant) / 정상(benign) **이진 분류**

**레이블 관례 조정**

sklearn 원본은 0=malignant, 1=benign으로 의료 진단 관례와 반대다.  
아래에서 레이블을 반전하여 표준 관례로 통일한다.

| 레이블 | 의미 | 분류 역할 |
|---|---|---|
| **1** | 악성 (malignant, 암) | **Positive** |
| **0** | 정상 (benign) | Negative |

- FN(암 환자를 정상으로 진단)이 FP보다 훨씬 위험 → **Recall** 우선 지표

### 1-1. 데이터 로딩 및 탐색

In [ ]:
cancer = load_breast_cancer()

# sklearn 원본: 0=malignant(악성), 1=benign(정상) — 의료 관례와 반대
cancer.target_names

In [ ]:
# 레이블 반전: 1=악성(Positive), 0=정상(Negative)
target_map     = {0: '정상', 1: '악성'}
class_names_kr = ['정상(benign)', '악성(malignant)']  # label 0, label 1 순서

df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['진단'] = 1 - cancer.target   # 반전: 1=악성, 0=정상
df['진단명'] = df['진단'].map(target_map)

print(f"데이터 크기: {df.shape}")
print(f"레이블 확인: 악성=1(Positive), 정상=0(Negative)")
print(f"클래스 분포:\n{df['진단명'].value_counts()}")
df.head()

In [ ]:
# 주요 통계량 (앞 6개 특성)
df.iloc[:, :6].describe().round(3)

### 1-2. 클래스별 특성 분포 시각화

악성/정상 그룹 간 특성값 차이를 박스플롯으로 확인한다.  
두 클래스가 잘 분리되는 특성일수록 분류에 유용하다.

In [ ]:
features_top6 = list(cancer.feature_names[:6])

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flat, features_top6):
    df.boxplot(column=feat, by='진단명', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='crimson', linewidth=2))
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')

fig.suptitle('악성 vs 정상 — 주요 특성 분포 비교', fontsize=13)
plt.tight_layout()
plt.show()

print("악성(malignant)은 전반적으로 특성값이 더 크고 분산도 높음")
print("→ 특성값만으로도 두 클래스가 상당히 분리됨")

### 1-3. 훈련/테스트 분할

`stratify=y` 옵션으로 분할 후에도 클래스 비율을 유지한다.

In [ ]:
X = cancer.data
y = 1 - cancer.target   # 반전: 악성=1(Positive), 정상=0(Negative)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"전체 샘플: {len(X)}")
print(f"훈련셋:   {X_tr.shape[0]}개  (악성(1) {(y_tr==1).sum()} / 정상(0) {(y_tr==0).sum()})")
print(f"테스트셋: {X_te.shape[0]}개  (악성(1) {(y_te==1).sum()} / 정상(0) {(y_te==0).sum()})")
print()
print("stratify=y → 분할 후에도 악성:정상 비율은 원본과 비슷")

---
## PART 2. K-최근접이웃 (KNN)

새로운 데이터와 **가장 가까운 K개의 이웃**을 찾아 다수결로 분류하는 알고리즘.

- **학습 단계 없음 (Lazy Learning)**: 훈련 데이터를 그대로 저장
- **거리 기반** → 변수 단위(scale)가 다르면 거리 왜곡 → **스케일 정규화 필수**
- **K 값 선택**: 작으면 과적합 / 크면 과소적합, 홀수 선택 권장 (동점 방지)
- 유클리드 거리 (기본값): $d = \sqrt{\sum_{i=1}^{n}(a_i - b_i)^2}$

### 2-1. 스케일 정규화의 필요성

KNN은 거리 기반 알고리즘이므로 변수 단위가 다르면 범위가 큰 변수가 거리를 좌우한다.

In [ ]:
print("정규화 전 — 특성별 범위:")
scale_df = pd.DataFrame({
    '특성':    cancer.feature_names[:6],
    '최솟값':  X_tr[:, :6].min(axis=0).round(3),
    '최댓값':  X_tr[:, :6].max(axis=0).round(3),
    '표준편차': X_tr[:, :6].std(axis=0).round(3)
})
print(scale_df.to_string(index=False))
print()
print("→ 'area'는 0~2500 범위, 'texture'는 0~40 범위")
print("→ 범위가 큰 area 변수가 거리 계산을 좌우 → 정규화 필수")

In [ ]:
# Min-Max 정규화 (0~1 범위로 변환)
scaler = MinMaxScaler()
X_tr_scaled = scaler.fit_transform(X_tr)   # 훈련셋 기준으로 fit
X_te_scaled = scaler.transform(X_te)        # 테스트셋은 transform만 (데이터 누수 방지)

print("정규화 후 — 특성별 범위 (훈련셋 기준):")
scale_after = pd.DataFrame({
    '특성':    cancer.feature_names[:6],
    '최솟값':  X_tr_scaled[:, :6].min(axis=0).round(3),
    '최댓값':  X_tr_scaled[:, :6].max(axis=0).round(3),
    '표준편차': X_tr_scaled[:, :6].std(axis=0).round(3)
})
print(scale_after.to_string(index=False))
print()
print("주의: scaler.fit()은 반드시 훈련셋에만!")
print("테스트셋에 fit_transform 하면 정보 누수(data leakage) 발생")

### 2-2. 정규화 전후 KNN 성능 비교

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

# 정규화 전
knn.fit(X_tr, y_tr)
acc_raw = knn.score(X_te, y_te)

# 정규화 후
knn.fit(X_tr_scaled, y_tr)
acc_scaled = knn.score(X_te_scaled, y_te)

print(f"KNN (K=5) 테스트 정확도")
print(f"  정규화 전: {acc_raw:.4f}")
print(f"  정규화 후: {acc_scaled:.4f}")
print(f"  향상폭:   +{(acc_scaled - acc_raw)*100:.1f}%p")
print()
print("→ 스케일 정규화만으로 정확도가 향상됨")

### 2-3. 최적 K 값 탐색

K가 너무 작으면 노이즈에 민감(과적합), 너무 크면 단순한 경계(과소적합).  
**교차검증(5-Fold CV)**으로 편향 없이 최적 K를 탐색한다.

- 선택 가이드: $\sqrt{N_{train}}$ 근처에서 시작 ($N_{train}$: 훈련 샘플 수), **홀수** 선택 권장

In [ ]:
k_range = range(1, 26, 2)   # 홀수만 (동점 방지)
train_accs, test_accs, cv_scores = [], [], []

for k in k_range:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_tr_scaled, y_tr)
    train_accs.append(knn_k.score(X_tr_scaled, y_tr))
    test_accs.append(knn_k.score(X_te_scaled,  y_te))
    cv = cross_val_score(knn_k, X_tr_scaled, y_tr, cv=5, scoring='accuracy')
    cv_scores.append(cv.mean())

best_k = list(k_range)[cv_scores.index(max(cv_scores))]
print(f"√N_train 근사값: {int(len(X_tr)**0.5)}  (N_train = 훈련 샘플 수 {len(X_tr)}개)")
print(f"5-폴드 CV 최고 정확도: {max(cv_scores):.4f}  →  최적 K = {best_k}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, train_accs, 'o-', color='crimson',    linewidth=2,   label='훈련 정확도')
ax.plot(k_range, test_accs,  's-', color='steelblue',  linewidth=2,   label='테스트 정확도')
ax.plot(k_range, cv_scores,  '^--', color='forestgreen', linewidth=1.5, label='5-폴드 CV 정확도')
ax.axvline(best_k, color='gray', linestyle='--', linewidth=1.5, label=f'최적 K = {best_k}')
ax.set_xlabel('K (이웃 수)', fontsize=12)
ax.set_ylabel('정확도', fontsize=12)
ax.set_title('K 값에 따른 KNN 성능 변화', fontsize=13)
ax.legend(fontsize=10)
ax.set_xticks(list(k_range))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2-4. 최적 K로 최종 KNN 모델

In [ ]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_tr_scaled, y_tr)
y_pred_knn = knn_best.predict(X_te_scaled)

print(f"최적 KNN 모델 (K={best_k})")
print(f"훈련 정확도: {knn_best.score(X_tr_scaled, y_tr):.4f}")
print(f"테스트 정확도: {knn_best.score(X_te_scaled, y_te):.4f}")
print()
print(classification_report(y_te, y_pred_knn, target_names=class_names_kr))

---
## PART 3. 랜덤 포레스트 (Random Forest)

**앙상블(Ensemble) 학습** — 여러 약한 학습기(Weak Learner)를 결합해 강한 학습기(Strong Learner)를 만드는 기법

랜덤 포레스트는 **배깅(Bagging)** 기반 앙상블 — 의사결정나무를 **이중 무작위성**으로 결합

| 무작위성 | 방법 | 효과 |
|---|---|---|
| **행(Row) 샘플링** | Bootstrap 복원추출 (≈ 63% 포함) | 다양한 훈련셋 생성 |
| **열(Column) 샘플링** | 각 분기마다 $\sqrt{p}$개 변수만 고려 | 트리 간 상관성 저하 |

**장점**: 과적합에 강함 / 스케일 정규화 불필요 / 특성 중요도 자동 산출  
**OOB 오차**: 복원추출로 선택되지 않은 샘플(≈37%)로 일반화 성능 추정 — 별도 검증셋 불필요

### 3-1. 랜덤 포레스트 기본 모델

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)   # 스케일 정규화 불필요
y_pred_rf = rf.predict(X_te)

print("랜덤 포레스트 (n_estimators=100)")
print(f"훈련 정확도: {rf.score(X_tr, y_tr):.4f}")
print(f"테스트 정확도: {rf.score(X_te, y_te):.4f}")
print()
print(classification_report(y_te, y_pred_rf, target_names=class_names_kr))

### 3-2. OOB (Out-of-Bag) 오차 확인

`oob_score=True`로 설정하면 훈련 과정에서 OOB 샘플로 자동 검증한다.

In [ ]:
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf_oob.fit(X_tr, y_tr)

print(f"OOB 점수 (훈련셋 내부 검증): {rf_oob.oob_score_:.4f}")
print(f"테스트 정확도:              {rf_oob.score(X_te, y_te):.4f}")
print()
print("→ OOB 점수 ≈ 테스트 정확도 → 별도 검증셋 없이 일반화 성능 신뢰 가능")
print("→ 테스트셋을 더 많은 훈련에 활용할 수 있는 장점")

### 3-3. 특성 중요도 (Feature Importance)

각 변수가 전체 불순도 감소에 기여한 비율 — 합 = 1.0  
**변수 선택**과 **해석**에 활용한다.

In [ ]:
fi = pd.Series(rf_oob.feature_importances_, index=cancer.feature_names)
fi_top10 = fi.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['crimson' if i < 3 else 'steelblue' for i in range(len(fi_top10))]
bars = ax.bar(fi_top10.index, fi_top10.values,
              color=colors, edgecolor='navy', alpha=0.85)

for bar, val in zip(bars, fi_top10.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('특성', fontsize=12)
ax.set_ylabel('중요도', fontsize=12)
ax.set_title('랜덤 포레스트 특성 중요도 (상위 10개)', fontsize=13)
ax.tick_params(axis='x', rotation=40)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"상위 3개 특성 (빨간색): {list(fi_top10.index[:3])}")
print(f"→ 전체 중요도의 {fi_top10[:3].sum():.1%} 차지")
print("→ 'worst area', 'worst radius' 등 세포 크기 관련 특성이 핵심")

### 3-4. 트리 수(n_estimators)에 따른 성능 변화

트리가 많을수록 예측이 안정되지만, 일정 수 이상에서 성능이 수렴한다.

In [ ]:
import time

n_trees_list = [10, 20, 50, 100, 200, 300, 500]
rf_train_scores, rf_test_scores, rf_oob_list, elapsed_list = [], [], [], []

for n in n_trees_list:
    t0 = time.time()
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42)
    rf_n.fit(X_tr, y_tr)
    elapsed = time.time() - t0

    rf_train_scores.append(rf_n.score(X_tr, y_tr))
    rf_test_scores.append(rf_n.score(X_te, y_te))
    rf_oob_list.append(rf_n.oob_score_)
    elapsed_list.append(elapsed)
    print(f"n_estimators={n:>4d}  →  {elapsed:.3f}초")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot(n_trees_list, rf_train_scores, 'o-',  color='crimson',     linewidth=2,   label='훈련 정확도')
ax.plot(n_trees_list, rf_test_scores,  's-',  color='steelblue',   linewidth=2,   label='테스트 정확도')
ax.plot(n_trees_list, rf_oob_list,     '^--', color='forestgreen', linewidth=1.5, label='OOB 정확도')
ax.set_xlabel('트리 수 (n_estimators)', fontsize=12)
ax.set_ylabel('정확도', fontsize=12)
ax.set_title('트리 수에 따른 랜덤 포레스트 성능 변화', fontsize=13)
ax.set_ylim(0.93, 1.01)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(n_trees_list, elapsed_list, 'D-', color='darkorange', linewidth=2)
for x, y in zip(n_trees_list, elapsed_list):
    ax.text(x, y + max(elapsed_list) * 0.01, f'{y:.3f}s',
            ha='center', va='bottom', fontsize=8)
ax.set_xlabel('트리 수 (n_estimators)', fontsize=12)
ax.set_ylabel('소요 시간 (초)', fontsize=12)
ax.set_title('트리 수에 따른 학습 시간', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("→ 트리 수 증가 → 분산 감소 → 성능 안정화")
print("→ 100~200개 이상에서 성능이 수렴하는 경향")
print("→ 너무 많은 트리는 계산 비용만 증가 (과적합은 없음)")

---
## PART 4. 모델 평가 지표

혼동 행렬의 4가지 원소(TP, TN, FP, FN)로 계산되는 핵심 지표들

| 지표 | 수식 | 우선 상황 |
|---|---|---|
| **정확도 (Accuracy)** | $(TP+TN)/전체$ | 균형 클래스 |
| **Precision** | $TP/(TP+FP)$ | FP 최소화 (스팸 필터) |
| **Recall** | $TP/(TP+FN)$ | FN 최소화 (암 진단) |
| **F1-Score** | $2 \times P \times R / (P+R)$ | 불균형 클래스 |
| **AUC** | ROC 곡선 면적 (0~1) | 임계값 무관 전체 성능 |

**이 데이터에서 우선 지표: Recall** — 악성(암)을 정상으로 잘못 진단(FN)이 더 치명적

### 4-1. 혼동 행렬 (Confusion Matrix)

분류 모델의 예측 결과를 실제 레이블과 대조하여 정리한 표

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (model_name, y_pred) in zip(axes, [
    (f'KNN (K={best_k})', y_pred_knn),
    ('랜덤 포레스트',     y_pred_rf)
]):
    cm = confusion_matrix(y_te, y_pred)
    # label 1=악성(Positive), label 0=정상(Negative)
    # confusion_matrix ravel() → (TN, FP, FN, TP) — sklearn 기본(positive=더 큰 label=1=악성)
    # cm[0,0]=TN(정상→정상), cm[0,1]=FP(정상→악성), cm[1,0]=FN(악성→정상), cm[1,1]=TP(악성→악성)
    tn, fp, fn, tp = cm.ravel()

    group_names  = [['TN', 'FP'], ['FN', 'TP']]
    group_counts = [[tn, fp], [fn, tp]]
    labels = np.array([[f'{name}\n{val}'
                        for name, val in zip(row_n, row_v)]
                       for row_n, row_v in zip(group_names, group_counts)])

    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues',
                xticklabels=['정상 예측(-)', '악성 예측(+)'],
                yticklabels=['정상 실제(-)', '악성 실제(+)'],
                ax=ax, cbar=False, linewidths=1,
                annot_kws={'size': 13})
    ax.set_title(f'{model_name} — 혼동 행렬', fontsize=12)
    ax.set_xlabel('예측 레이블', fontsize=11)
    ax.set_ylabel('실제 레이블', fontsize=11)

plt.suptitle('KNN vs 랜덤 포레스트 — 혼동 행렬 비교 (Positive = 악성(1))', fontsize=13)
plt.tight_layout()
plt.show()

print("FN = 악성 환자를 정상으로 진단 → 2종 오류 (Miss) — 더 위험")
print("FP = 정상인을 악성으로 진단   → 1종 오류 (False Alarm)")

### 4-2. 분류 평가 지표 계산 및 비교

In [ ]:
models_eval = {
    f'KNN (K={best_k})': y_pred_knn,
    '랜덤 포레스트':     y_pred_rf,
}

rows = []
for name, y_pred in models_eval.items():
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    rows.append({
        '모델':      name,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        '정확도':    round(accuracy_score(y_te, y_pred),  4),
        'Precision': round(precision_score(y_te, y_pred), 4),
        'Recall':    round(recall_score(y_te, y_pred),    4),
        'F1-Score':  round(f1_score(y_te, y_pred),        4),
    })

metrics_df = pd.DataFrame(rows).set_index('모델')
print("평가 지표 비교 (Positive = 악성(1))")
metrics_df

In [ ]:
best_recall_model = metrics_df['Recall'].idxmax()
print(f"Recall 기준 최우수 모델: {best_recall_model}")
print()
print("암 진단에서 FN(악성을 정상으로 진단)이 더 치명적")
print("→ 임계값을 낮추면 Recall ↑, Precision ↓ (더 많이 악성으로 예측)")
print("→ 도메인에 따라 임계값 조정으로 FN을 줄이는 전략 필요")

### 4-3. Precision-Recall Trade-off

임계값(threshold) 조정에 따라 Precision과 Recall은 **반비례 관계**
- 임계값 낮춤 → 더 많이 악성(+)으로 예측 → **Recall ↑, Precision ↓**
- 임계값 높임 → 더 신중하게 악성(+)으로 예측 → **Precision ↑, Recall ↓**

In [ ]:
# 악성(1=Positive)에 대한 예측 확률
y_prob_knn = knn_best.predict_proba(X_te_scaled)[:, 1]
y_prob_rf  = rf_oob.predict_proba(X_te)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, y_prob) in zip(axes, [
    (f'KNN (K={best_k})', y_prob_knn),
    ('랜덤 포레스트',     y_prob_rf)
]):
    precisions, recalls, thresholds = precision_recall_curve(y_te, y_prob)  # pos_label=1 기본값

    ax.plot(thresholds, precisions[:-1], color='crimson',   linewidth=2,   label='Precision')
    ax.plot(thresholds, recalls[:-1],    color='steelblue', linewidth=2,   linestyle='--', label='Recall')

    # F1 최대 지점 (최적 임계값)
    denom = precisions[:-1] + recalls[:-1]
    f1_arr = np.where(denom > 0,
                      2 * precisions[:-1] * recalls[:-1] / denom, 0)
    best_idx = f1_arr.argmax()
    best_thr = thresholds[best_idx]
    ax.axvline(best_thr, color='forestgreen', linestyle=':', linewidth=1.8,
               label=f'F1 최대 임계값 = {best_thr:.2f}')
    ax.axvline(0.5, color='orange', linestyle='--', linewidth=1.2,
               label='기본 임계값 = 0.50', alpha=0.7)

    ax.set_xlabel('임계값 (Threshold)', fontsize=11)
    ax.set_ylabel('지표값', fontsize=11)
    ax.set_title(f'{name}\nPrecision-Recall Trade-off', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Precision-Recall Trade-off — 임계값 조정 효과', fontsize=13)
plt.tight_layout()
plt.show()

print("암 진단: 임계값을 낮춰 Recall 우선 → 악성 환자를 최대한 잡아냄")
print("스팸 필터: 임계값을 높여 Precision 우선 → 정상 메일을 스팸으로 분류하지 않음")

### 4-4. ROC 곡선 & AUC

다양한 임계값에서 **TPR(Recall)과 FPR(1−특이도)**의 관계를 시각화한 곡선

$$TPR = \frac{TP}{TP + FN} = \text{Recall}, \quad FPR = \frac{FP}{FP + TN} = 1 - \text{특이도}$$

**AUC (Area Under Curve)**: 임계값과 독립적인 전체 성능 요약
- AUC = 1.0: 완벽한 분류기
- AUC = 0.5: 무작위 분류기 (기준선)
- AUC > 0.8: 일반적으로 우수한 성능

In [ ]:
roc_curve(y_te, y_prob)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# 무작위 기준선
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='무작위 분류 (AUC = 0.5)', alpha=0.7)

colors_roc = ['steelblue', 'crimson']
for (name, y_prob), color in zip([
    (f'KNN (K={best_k})', y_prob_knn),
    ('랜덤 포레스트',     y_prob_rf)
], colors_roc):
    fpr_arr, tpr_arr, _ = roc_curve(y_te, y_prob)   # pos_label=1(악성) 기본값
    roc_auc = auc(fpr_arr, tpr_arr)
    ax.plot(fpr_arr, tpr_arr, color=color, linewidth=2.5,
            label=f'{name}  (AUC = {roc_auc:.3f})')

ax.set_xlabel('FPR (1 − 특이도)', fontsize=12)
ax.set_ylabel('TPR (재현율)', fontsize=12)
ax.set_title('ROC 곡선 — KNN vs 랜덤 포레스트', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("AUC가 클수록 임계값과 무관하게 전체적으로 우수한 분류기")
print("AUC > 0.8 → 일반적으로 우수한 성능")

### 4-5. 모델 종합 비교

In [ ]:
summary_rows = []
for name, y_pred, y_prob in [
    (f'KNN (K={best_k})', y_pred_knn, y_prob_knn),
    ('랜덤 포레스트',     y_pred_rf,  y_prob_rf),
]:
    summary_rows.append({
        '모델':      name,
        '정확도':    f"{accuracy_score(y_te, y_pred):.4f}",
        'Recall':    f"{recall_score(y_te, y_pred):.4f}",
        'Precision': f"{precision_score(y_te, y_pred):.4f}",
        'F1-Score':  f"{f1_score(y_te, y_pred):.4f}",
        'AUC':       f"{roc_auc_score(y_te, y_prob):.4f}",
    })

summary_df = pd.DataFrame(summary_rows).set_index('모델')
print("종합 성능 비교표")
summary_df

In [ ]:
metric_cols = ['정확도', 'Recall', 'Precision', 'F1-Score', 'AUC']
summary_num = summary_df[metric_cols].astype(float)

x = np.arange(len(metric_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
for i, (model_name, row) in enumerate(summary_num.iterrows()):
    bars = ax.bar(x + i * width, row.values, width,
                  label=model_name,
                  color=['steelblue', 'crimson'][i],
                  alpha=0.85, edgecolor='navy')
    for bar, val in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(metric_cols, fontsize=11)
ax.set_ylabel('점수', fontsize=12)
ax.set_ylim(0.85, 1.03)
ax.set_title('KNN vs 랜덤 포레스트 — 주요 평가 지표 비교', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 요약

### 핵심 개념

| 주제 | 개념 | 주요 함수/속성 |
|---|---|---|
| **KNN** | K개 최근접 이웃의 다수결 분류 | `KNeighborsClassifier` |
| 스케일 정규화 | 거리 기반 → Min-Max 또는 표준화 필수 | `MinMaxScaler` |
| K 선택 | $\sqrt{N_{train}}$ 근처, 홀수, 교차검증으로 탐색 ($N_{train}$: 훈련 샘플 수) | `cross_val_score` |
| **랜덤 포레스트** | 의사결정나무의 배깅 앙상블 | `RandomForestClassifier` |
| 이중 무작위성 | 행(Bootstrap) + 열(무작위 특성 선택) | — |
| OOB 오차 | 별도 검증셋 없이 일반화 성능 추정 | `oob_score=True` |
| 특성 중요도 | 불순도 감소 기여 비율, 합 = 1 | `feature_importances_` |
| **혼동 행렬** | TP / TN / FP / FN | `confusion_matrix` |
| 정확도 | $(TP+TN)/전체$ — 균형 클래스 | `accuracy_score` |
| Precision | $TP/(TP+FP)$ — FP 최소화 | `precision_score` |
| Recall | $TP/(TP+FN)$ — FN 최소화 (암 진단) | `recall_score` |
| F1-Score | Precision·Recall 조화평균 — 불균형 클래스 | `f1_score` |
| ROC / AUC | 임계값 무관 전체 성능 요약 | `roc_curve`, `auc` |

### 실습 결과 요약

- **데이터**: 유방암 진단 (569샘플, 30특성, 이진 분류 — 악성/정상)
- **KNN**: 스케일 정규화 후 성능 향상 확인; 교차검증으로 최적 K 탐색
- **랜덤 포레스트**: 정규화 불필요; OOB 오차로 안정적 성능 추정; `worst area`, `worst radius` 등 세포 크기·형태 관련 특성이 분류에 핵심
- **모델 평가**: 암 진단에서 Recall이 핵심 지표 — FN(악성 환자를 정상으로 진단) 최소화가 목표

> **알고리즘 선택 기준**  
> 해석 중요 → 의사결정나무  
> 소규모·단순 → KNN  
> **성능 우선·표 데이터 → 랜덤 포레스트**
>
> **임계값 전략**  
> 스팸 메일 → Precision 우선 (임계값 ↑)  
> 암 진단 → Recall 우선 (임계값 ↓)  
> 둘 다 중요 → F1-Score